In [14]:
import pandas as pd
import os
import glob

pd.set_option('future.no_silent_downcasting', True)

In [21]:
# Directory containing the CSV files
directory = r"/home/alicia_bassiere/python_projects/cem_bb/residual_load"

# Read all CSV files in the directory
csv_files = [file for file in os.listdir(directory) if file.endswith('.csv')]
gen_files = {file: pd.read_csv(os.path.join(directory, file)).iloc[:, 1:] for file in csv_files}

for key in gen_files:
    gen_df = gen_files[key]
    gen_df.columns = [col.replace(" - Actual Aggregated [MW]", "").lower() for col in gen_df.columns]
    
    # Rename the "MTU" column to "time"
    gen_df.rename(columns={"mtu": "time"}, inplace=True)

    # Convert the "time" column to datetime and set it as the index
    gen_df['time'] = pd.to_datetime(gen_df['time'].str.split(' - ').str[0], format='%d.%m.%Y %H:%M')
    gen_df.set_index('time', inplace=True)
    gen_df.sort_index(ascending=True, inplace=True)
    
    # Resample the data at an hourly level
    gen_df = gen_df.resample('h').max()
    # Remove February 29th if it exists
    gen_df = gen_df[~((gen_df.index.month == 2) & (gen_df.index.day == 29))]
    gen_df['energy storage'] = gen_df['energy storage'].replace("n/e", 0).astype(float, errors='ignore')
    gen_df['marine'] = gen_df['marine'].replace("n/e", 0).astype(float, errors='ignore')


    gen_df['additional_generation'] = gen_df['energy storage'] + gen_df['geothermal'] + gen_df['biomass'] + gen_df['hydro water reservoir']  
    + gen_df['hydro pumped storage'] + gen_df['hydro run-of-river and poundage'] + gen_df['marine']\
        + gen_df['other'] + gen_df['other renewable'] + gen_df['waste'] + gen_df['wind offshore']

    
    gen_files[key] = gen_df


gen_files = {str(df.index.year[0]): df for df in gen_files.values()}
excess_demand = {key: df[['additional_generation']] for key, df in gen_files.items()}

for key, load_df in excess_demand.items():
    # Remove the year from the datetime index
    load_df.index = load_df.index.strftime('%m-%d %H:%M')
    
    # Replace the name of the column "additional_generation" with the key
    excess_demand[key] = load_df.rename(columns={"additional_generation": key}).copy()

# Merge all dataframes in excess_demand based on the datetime index
excess_demand = pd.concat(excess_demand.values(), axis=1)
excess_demand = excess_demand[sorted(excess_demand.columns)]

# Duplicate "2021" column to "favorable" and "adverse"
excess_demand['favorable'] = excess_demand['2021']
excess_demand['adverse'] = excess_demand['2021']

# Drop the columns "2020", "2021", and "2022"
excess_demand = excess_demand.drop(columns=['2020', '2021', '2022'])

if excess_demand.isnull().values.any():
    print("NaN values found in excess_demand at the following locations:")
    print(excess_demand[excess_demand.isnull().any(axis=1)])
    # Interpolate missing values
    excess_demand.interpolate(method='linear', inplace=True)
    print("NaN values have been interpolated.")
else:
    print("No NaN values found in excess_demand.")
    
# Display the merged dataframe
# Save the merged dataframe as a CSV file
output_path = os.path.expanduser("~/python_projects/cem_bb/inputs/excess_demand.csv")
excess_demand.to_csv(output_path)


NaN values found in excess_demand at the following locations:
               2015    2016    2017    2018    2019  favorable  adverse
time                                                                   
03-25 02:00  4654.0  4923.0  4831.0     NaN  4735.0     4095.0   4095.0
03-26 02:00  4606.0  4898.0     NaN  4501.0  4793.0     4262.0   4262.0
03-27 02:00  4668.0     NaN  4822.0  4763.0  4804.0     4397.0   4397.0
03-28 02:00  4637.0  4909.0  4789.0  4755.0  4696.0        NaN      NaN
03-29 02:00     NaN  4858.0  4793.0  4770.0  4656.0     4433.0   4433.0
03-31 02:00  4590.0  4809.0  4682.0  4731.0     NaN     4369.0   4369.0
NaN values have been interpolated.


In [16]:
# Define the path

folder_load = r'/home/alicia_bassiere/python_projects/cem_bb/load'

load_files = glob.glob(os.path.join(folder_load, "*.csv"))

# Read all CSV files into a list of DataFrames
dataframes_load = {os.path.basename(file): pd.read_csv(file) for file in load_files}

for key in dataframes_load:
    load_df = dataframes_load[key]
    load_df['time'] = pd.to_datetime(load_df.iloc[:, 0].str.split(' - ').str[0], format='%d.%m.%Y %H:%M')
    load_df.set_index('time', inplace=True)
    load_df.sort_index(ascending=True, inplace=True)
    load_df = load_df.resample('h').max()
    load_df = load_df[['Actual Total Load [MW] - Germany (DE)']].rename(columns={"Actual Total Load [MW] - Germany (DE)": "load"})
    dataframes_load[key] = load_df

dataframes_load = {str(df.index.year[0]): df for df in dataframes_load.values()}

load = {key: df[['load']] for key, df in dataframes_load.items()}

for key, load_df in load.items():
    load_df.index = load_df.index.strftime('%m-%d %H:%M')
    load_df = load_df[~((load_df.index.str.startswith('02-29')))]
    load[key] = load_df.rename(columns={"load": key}).copy()

load = pd.concat(load.values(), axis=1)
load = load[sorted(load.columns)]


In [17]:
## Creation of scenarios

load = load.drop(columns=['2020', '2022'])

# Define winter months
winter_months = ['12', '01', '02']

# Create the "favorable" column
load['favorable'] = load['2021']
load.loc[load.index.str[:2].isin(winter_months), 'favorable'] *= 0.88

# Create the "adverse" column
load['adverse'] = load['2021']
load.loc[load.index.str[:2].isin(winter_months), 'adverse'] *= 1.12

load = load.drop(columns=['2021'])

# Interpolate missing values in the load DataFrame
load.interpolate(method='linear', inplace=True)

output_path = os.path.expanduser("~/python_projects/cem_bb/inputs/load_scenarios.csv")
load.to_csv(output_path)

load_excess_demand = load - excess_demand

In [18]:
# Calculate statistics for load
load_stats = load.agg(['mean', 'min', 'max'])

# Calculate statistics for load_excess_demand
load_excess_demand_stats = load_excess_demand.agg(['mean', 'min', 'max'])

# Display the statistics
print("Load Statistics:")
print(load_stats)

print("\nLoad Excess Demand Statistics:")
print(load_excess_demand_stats)

Load Statistics:
              2015          2016          2017          2018          2019  \
mean  58086.373973  58112.348916  58607.365639  58866.259932  57495.487957   
min   34700.000000  35846.000000  35145.000000  36379.000000  34246.000000   
max   78068.000000  80179.000000  79817.000000  81260.000000  77934.000000   

         favorable       adverse  
mean  56472.205664  60140.104952  
min   34367.520000  36768.000000  
max   81614.000000  89522.720000  

Load Excess Demand Statistics:
              2015          2016          2017          2018          2019  \
mean  53387.643567  53308.317502  53859.581345  54144.354721  52665.980363   
min   29874.000000  31245.000000  30584.000000  31845.000000  29602.000000   
max   73168.000000  75271.000000  74866.000000  76291.000000  72584.000000   

         favorable       adverse  
mean  51819.733317  55488.051362  
min   29800.520000  32348.000000  
max   76987.000000  84909.720000  


In [19]:
# Generate LaTeX table for the load DataFrame
latex_table = "\\multirow{3}{*}{\\textbf{Load_new}\\tnote{a}} & Mean & " + " & ".join(f"{(v / 1000):.1f}" for v in load_stats.loc['mean']) + " \\\\\n"
latex_table += "     & Min & " + " & ".join(f"{(v / 1000):.1f}" for v in load_stats.loc['min']) + " \\\\\n"
latex_table += "     & Max & " + " & ".join(f"{(v / 1000):.1f}" for v in load_stats.loc['max']) + " \\\\"

# Print the LaTeX table
print(latex_table)

# Generate LaTeX table for the load_excess_demand_stats DataFrame
latex_table = "\\multirow{3}{*}{\\textbf{Load_new (Net)}\\tnote{b}} & Mean & " + " & ".join(f"{(v / 1000):.1f}" for v in load_excess_demand_stats.loc['mean']) + " \\\\\n"
latex_table += "     & Min & " + " & ".join(f"{(v / 1000):.1f}" for v in load_excess_demand_stats.loc['min']) + " \\\\\n"
latex_table += "     & Max & " + " & ".join(f"{(v / 1000):.1f}" for v in load_excess_demand_stats.loc['max']) + " \\\\"

# Print the LaTeX table
print(latex_table)

\multirow{3}{*}{\textbf{Load_new}\tnote{a}} & Mean & 58.1 & 58.1 & 58.6 & 58.9 & 57.5 & 56.5 & 60.1 \\
     & Min & 34.7 & 35.8 & 35.1 & 36.4 & 34.2 & 34.4 & 36.8 \\
     & Max & 78.1 & 80.2 & 79.8 & 81.3 & 77.9 & 81.6 & 89.5 \\
\multirow{3}{*}{\textbf{Load_new (Net)}\tnote{b}} & Mean & 53.4 & 53.3 & 53.9 & 54.1 & 52.7 & 51.8 & 55.5 \\
     & Min & 29.9 & 31.2 & 30.6 & 31.8 & 29.6 & 29.8 & 32.3 \\
     & Max & 73.2 & 75.3 & 74.9 & 76.3 & 72.6 & 77.0 & 84.9 \\
